In [1]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

column_types = {
"osm_id": "str",
"addr_street": "str",
"addr_number": "str",
"grupa_parter": "str",
"funkcja_parter": "str",
"grupa_pietra": "str",
"funkcja_pietra": "str",
"liczba_kondygnacji": "Int64",
"mieszana": "boolean",
"rok_budowy_szac": "Int64",
"styl_arch": "str",
"stan_tech": "Int64",
"wysokosc_max": "float64",
"zabytek": "str"
}

In [2]:
data_gdynia = pd.read_csv("../../data/raw/srodmiescie_raw.csv",
                          sep=";", index_col = 0, dtype = column_types)

data_clean = data_gdynia.dropna(thresh = 3)
data_gdynia.head()

,osm_id,addr_street,addr_number,grupa_parter,funkcja_parter,grupa_pietra,funkcja_pietra,liczba_kondygnacji,mieszana,rok_budowy_szac,styl_arch,stan_tech,wysokosc_max,zabytek,zrodlo,pewnosc,data_sprawdzenia
id,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,Hotel,NaN,Hotel,7,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
2,890729731,Świętego Piotra,6D,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
3,890292040,Węglowa,22,NaN,Centrum Handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
4,890729732,Świętego Piotra,6,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
5,136885205,Tadeusza Wendy,15,NaN,Centrum Medyczne,NaN,Centrum Medyczne,6,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,2.0,27.09.2026


In [ ]:
max_func_per_place = ((data_clean["funkcja_parter"].str.count(',')+1).max())
number_of_all_func = (data_clean["funkcja_parter"].str.count(',')+1).sum()
print(f'max_func_per_place: {max_func_per_place}')
print(f'number_of_all_func: {number_of_all_func}')

max_func_per_place: 8.0
number_of_all_func: 1449


In [4]:
data_extract = (
    data_clean
    .assign(funkcja_parter = lambda d: d['funkcja_parter'].str.replace('_',' ').str.lower().str.split(pat = ',') )
    .explode('funkcja_parter')
    .assign(funkcja_parter = lambda d: d['funkcja_parter'].str.strip())
)

In [5]:
assert data_extract["funkcja_parter"].isna().sum() == 0

In [6]:
assert (data_extract["funkcja_parter"] == "").sum() == 0

In [7]:
assert len(data_extract.index) == number_of_all_func

In [8]:
data_comparison =(
    data_extract
    .groupby('funkcja_parter')
    .funkcja_parter.count()
    .reset_index(name = 'count')
    .rename(columns= {'funkcja_parter':'functions'})
)

In [9]:
engine = create_engine("postgresql+psycopg://postgres@localhost:5432/Dataset_gdynia")
query = Path("../../sql/functions_long.sql").read_text(encoding = 'utf-8')

data_gdynia_sql = (
    pd.read_sql(query, engine)
    .sort_values('functions')
    .reset_index(drop = True)
)

check if the sql export is the same as pandas

In [10]:
pd.testing.assert_frame_equal(data_gdynia_sql,data_comparison)